# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
# Colab: nếu chưa có repo, bỏ comment 2 dòng dưới, rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git
# REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

REPO_ROOT = os.path.abspath(REPO_ROOT)
OUT_DIR = os.path.abspath(OUT_DIR)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
device_name = torch.cuda.get_device_name(0) if device.type == "cuda" else device.type
print(f"PyTorch {torch.__version__}; device={device} ({device_name})")
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "results"), exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
split_env = os.environ.copy()
split_env["PYTHONIOENCODING"] = "utf-8"
subprocess.run([sys.executable, os.path.join(REPO_ROOT, "scripts", "split_data.py")], cwd=REPO_ROOT, env=split_env, check=True)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=os.path.join(REPO_ROOT, "data", "processed"))
for name in ("X_tr", "y_tr", "X_val", "y_val", "X_eval", "y_eval"):
    print(name, tuple(data[name].shape), data[name].dtype, data[name].device)
majority = int(torch.bincount(data["y_tr"], minlength=7).argmax())
majority_acc = (data["y_val"] == majority).float().mean().item()
numeric_mean = data["X_tr"][:, :10].mean(dim=0).detach().cpu().numpy()
numeric_std = data["X_tr"][:, :10].std(dim=0, unbiased=False).detach().cpu().numpy()
print(f"majority class={majority}, val accuracy={majority_acc:.4f}")
print("train numeric mean:", np.round(numeric_mean, 4))
print("train numeric std :", np.round(numeric_std, 4))
assert data["X_tr"].shape == (371847, 54)
assert data["X_val"].shape == (92962, 54)
assert data["X_eval"].shape == (116203, 54)
assert data["y_tr"].shape == (371847,) and data["y_tr"].dtype == torch.int64
assert data["y_val"].shape == (92962,) and data["y_val"].dtype == torch.int64
assert data["y_eval"].shape == (116203,) and data["y_eval"].dtype == torch.int64
assert abs(majority_acc - 0.4876) < 0.002
assert np.allclose(numeric_mean, 0.0, atol=2e-3)
assert np.allclose(numeric_std, 1.0, atol=2e-3)

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
torch.manual_seed(7)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
print("M-base parameters:", n_params)
assert n_params == EXPECTED_PARAMS[(256, 128)] == 47879
probe = torch.randn(8, 54, device=device)
logits = model(probe)
print("probe shape -> logits shape:", tuple(probe.shape), "->", tuple(logits.shape))
assert logits.shape == (8, 7) and logits.dtype == torch.float32
model.eval()
with torch.no_grad():
    step0_loss = torch.nn.functional.cross_entropy(model(data["X_val"]), data["y_val"]).item()
print(f"step-0 validation cross-entropy={step0_loss:.4f}; ln(7)={np.log(7):.4f}")
print("activation std after Linear layers:", activation_stats(model, data["X_val"][:256]))
torch.manual_seed(11)
tiny_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
tiny_optimizer = torch.optim.Adam(tiny_model.parameters(), lr=0.02)
tiny_X, tiny_y = data["X_tr"][:20], data["y_tr"][:20]
tiny_curve = []
for _ in range(300):
    tiny_optimizer.zero_grad(set_to_none=True)
    tiny_loss = torch.nn.functional.cross_entropy(tiny_model(tiny_X), tiny_y)
    tiny_loss.backward()
    tiny_optimizer.step()
    tiny_curve.append(float(tiny_loss.detach().cpu()))
tiny_model.eval()
with torch.no_grad():
    tiny_pred = tiny_model(tiny_X).argmax(dim=1)
tiny_acc = (tiny_pred == tiny_y).float().mean().item()
print(f"20-sample overfit: loss {tiny_curve[0]:.4f} -> {tiny_curve[-1]:.6f}, accuracy={tiny_acc:.1%}")
assert tiny_curve[-1] < 0.01 and tiny_acc == 1.0
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot(tiny_curve)
plt.xlabel("step"); plt.ylabel("cross-entropy"); plt.title("Part 1: overfit 20 samples")
plt.grid(alpha=0.25); plt.show()
gradient_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
gradient_model.zero_grad(set_to_none=True)
gradient_loss = torch.nn.functional.cross_entropy(gradient_model(data["X_val"][:64]), data["y_val"][:64])
gradient_loss.backward()
grad_norms = {name: float(parameter.grad.norm().item()) for name, parameter in gradient_model.named_parameters()}
print("gradient norms:", grad_norms)
assert all(parameter.grad is not None for parameter in gradient_model.parameters())
assert all(np.isfinite(value) and value > 0.0 for value in grad_norms.values())

**Nhận xét Part 1:** loss bước 0 được in trực tiếp ở ô trên và được so với `ln(7) = 1.9459`; giá trị có thể cao hơn mốc vì He initialization tạo logits chưa đồng đều trước khi học. Phép thử 20 mẫu phải đưa loss gần 0 và accuracy lên 100%, chứng tỏ forward, nhãn và cập nhật tham số hoạt động. Các gradient sau `backward()` được in theo từng tham số; tất cả phải hữu hạn và khác 0.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
lr_candidates = [0.01, 0.03, 0.1]
lr_probe_results = []
for candidate in lr_candidates:
    probe_cfg = {**DEFAULT_CFG, "exp_id": f"lr-probe-{candidate:g}", "lr": candidate, "epochs": 3, "seed": 1}
    probe = run_experiment(probe_cfg, data)
    lr_probe_results.append(probe)
    print(f"lr={candidate:g} -> val_acc={probe['summary']['val_acc']:.4f}, val_macro_f1={probe['summary']['val_macro_f1']:.4f}")
best_lr_result = max(lr_probe_results, key=lambda result: result["summary"]["val_macro_f1"])
best_lr = float(best_lr_result["cfg"]["lr"])
print("selected learning rate by validation macro-F1:", best_lr)
cfg = {**DEFAULT_CFG, "lr": best_lr}
baseline_results = []
for seed in (1, 2, 3):
    exp_cfg = {**cfg, "seed": seed, "exp_id": f"base-s{seed}", "description": f"Baseline M-base, seed {seed}"}
    result = run_experiment(exp_cfg, data)
    baseline_results.append(result)
    exp_id = exp_cfg["exp_id"]
    save_result(result, os.path.join(OUT_DIR, "results"))
    plot_run(result, os.path.join(OUT_DIR, "figures", f"{exp_id}.png"))
    print(exp_id, result["summary"])
seed_f1 = np.array([r["summary"]["val_macro_f1"] for r in baseline_results])
seed_acc = np.array([r["summary"]["val_acc"] for r in baseline_results])
print(f"baseline val_macro_f1={seed_f1.mean():.4f} ± {seed_f1.std(ddof=1):.4f}; 2σ={2 * seed_f1.std(ddof=1):.4f}")
print(f"baseline val_acc={seed_acc.mean():.4f} ± {seed_acc.std(ddof=1):.4f}")

**Nhận xét baseline:** dùng cùng phép tách validation và ba seed để tách thay đổi do cấu hình khỏi nhiễu khởi tạo. Đường cong trong các ảnh `figures/base-s*.png` cho biết loss còn giảm hay bắt đầu quá khớp; so sánh `val_acc` với mốc đoán lớp đa số `0.4876`, và dùng `2σ` in ở ô trên để đánh giá độ lớn của chênh lệch giữa các seed.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary